# What is "chunking"?

In [2]:
# Cài đặt thư viện
# %pip install langchain-text-splitters
# %pip install langchain-core

In [3]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

text = """
A database is a collection of related data. By data, we mean known facts
that can be recorded and that have implicit meaning.

A database management system (DBMS) is a computerized system that enables
users to create and maintain a database. The DBMS is a general-purpose
software system that facilitates the processes of defining, constructing,
manipulating, and sharing databases among various users and applications.

Defining a database involves specifying the data types, structures, and
constraints of the data to be stored in the database.

Constructing the database is the process of storing the data on some
storage medium that is controlled by the DBMS.

Manipulating a database includes querying the database, updating the data,
and generating reports.
"""

print("Characters:", len(text))
print("Words:", len(text.split()))

Characters: 765
Words: 117


## CharacterTextSplitter

- Lý do đoạn 1 là một chunk mà đoạn 3+4 được gộp lại thành một chunk là vì cơ chế của CharacterTextSplitter với separator="\n\n" có đơn vị nhỏ nhất là paragraph, cộng với việc chunk_size=250.
- VD: Đoạn 1 dài 125, khi cộng thêm đoạn 2 dài 292 sẽ lớn hơn 250. Do đó đoạn 1 sẽ là một chunk. Đoạn 2 dài 292, khi cộng thêm đoạn 3 hay sau đó thì cũng lớn hơn 250 nên đoạn 2 là một chunk. Nhưng đoạn 3 và đoạn 4 cộng lại bé hơn 250, khi cộng thêm đoạn 5 thì lớn hơn 250. Do đó đoạn 3+4 là một chunk và đoạn 5 là một chunk.

In [4]:
splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=250,
    chunk_overlap=50,
    length_function=len
)

chunks = splitter.split_text(text)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print("=" * 80)
    print(f"CHUNK {i}")
    print("Length:", len(chunk))
    print(chunk)

Created a chunk of size 292, which is longer than the specified 250


Number of chunks: 4
CHUNK 0
Length: 125
A database is a collection of related data. By data, we mean known facts
that can be recorded and that have implicit meaning.
CHUNK 1
Length: 292
A database management system (DBMS) is a computerized system that enables
users to create and maintain a database. The DBMS is a general-purpose
software system that facilitates the processes of defining, constructing,
manipulating, and sharing databases among various users and applications.
CHUNK 2
Length: 242
Defining a database involves specifying the data types, structures, and
constraints of the data to be stored in the database.

Constructing the database is the process of storing the data on some
storage medium that is controlled by the DBMS.
CHUNK 3
Length: 98
Manipulating a database includes querying the database, updating the data,
and generating reports.


In [5]:
for size in [150, 250, 400]:

    splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=size,
        chunk_overlap=30,
        length_function=len
    )

    chunks = splitter.split_text(text)
    print("=" * 100)
    print("CHUNK SIZE =", size)
    print("NUMBER OF CHUNKS =", len(chunks))

    for i, chunk in enumerate(chunks):
        print(
            f"\nChunk {i} | "
            f"length={len(chunk)}"
        )
        print(chunk)

Created a chunk of size 292, which is longer than the specified 150
Created a chunk of size 292, which is longer than the specified 250


CHUNK SIZE = 150
NUMBER OF CHUNKS = 5

Chunk 0 | length=125
A database is a collection of related data. By data, we mean known facts
that can be recorded and that have implicit meaning.

Chunk 1 | length=292
A database management system (DBMS) is a computerized system that enables
users to create and maintain a database. The DBMS is a general-purpose
software system that facilitates the processes of defining, constructing,
manipulating, and sharing databases among various users and applications.

Chunk 2 | length=125
Defining a database involves specifying the data types, structures, and
constraints of the data to be stored in the database.

Chunk 3 | length=115
Constructing the database is the process of storing the data on some
storage medium that is controlled by the DBMS.

Chunk 4 | length=98
Manipulating a database includes querying the database, updating the data,
and generating reports.
CHUNK SIZE = 250
NUMBER OF CHUNKS = 4

Chunk 0 | length=125
A database is a collection of rel

## RecursiveCharacterTextSplitter

In [6]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=50,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print("Number of chunks:", len(recursive_chunks))

for i, chunk in enumerate(recursive_chunks):
    print("=" * 80)
    print(f"CHUNK {i} | "f"length={len(chunk)}")
    print(chunk)

Number of chunks: 6
CHUNK 0 | length=125
A database is a collection of related data. By data, we mean known facts
that can be recorded and that have implicit meaning.
CHUNK 1 | length=144
A database management system (DBMS) is a computerized system that enables
users to create and maintain a database. The DBMS is a general-purpose
CHUNK 2 | length=147
software system that facilitates the processes of defining, constructing,
manipulating, and sharing databases among various users and applications.
CHUNK 3 | length=125
Defining a database involves specifying the data types, structures, and
constraints of the data to be stored in the database.
CHUNK 4 | length=115
Constructing the database is the process of storing the data on some
storage medium that is controlled by the DBMS.
CHUNK 5 | length=98
Manipulating a database includes querying the database, updating the data,
and generating reports.


## Overlap Experiment

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=50,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):

    print("=" * 80)
    print(f"CHUNK {i}")
    print(repr(chunk))

CHUNK 0
'A database is a collection of related data. By data, we mean known facts\nthat can be recorded and that have implicit meaning.'
CHUNK 1
'A database management system (DBMS) is a computerized system that enables\nusers to create and maintain a database. The DBMS is a general-purpose'
CHUNK 2
'software system that facilitates the processes of defining, constructing,\nmanipulating, and sharing databases among various users and applications.'
CHUNK 3
'Defining a database involves specifying the data types, structures, and\nconstraints of the data to be stored in the database.'
CHUNK 4
'Constructing the database is the process of storing the data on some\nstorage medium that is controlled by the DBMS.'
CHUNK 5
'Manipulating a database includes querying the database, updating the data,\nand generating reports.'


In [8]:
for i in range(len(chunks) - 1):

    print("=" * 80)
    print("END OF CHUNK", i)
    print(chunks[i][-80:], "\n")
    print("START OF CHUNK", i + 1)
    print(chunks[i + 1][:80])

END OF CHUNK 0
y data, we mean known facts
that can be recorded and that have implicit meaning. 

START OF CHUNK 1
A database management system (DBMS) is a computerized system that enables
users 
END OF CHUNK 1
t enables
users to create and maintain a database. The DBMS is a general-purpose 

START OF CHUNK 2
software system that facilitates the processes of defining, constructing,
manipu
END OF CHUNK 2
cting,
manipulating, and sharing databases among various users and applications. 

START OF CHUNK 3
Defining a database involves specifying the data types, structures, and
constrai
END OF CHUNK 3
ata types, structures, and
constraints of the data to be stored in the database. 

START OF CHUNK 4
Constructing the database is the process of storing the data on some
storage med
END OF CHUNK 4
ocess of storing the data on some
storage medium that is controlled by the DBMS. 

START OF CHUNK 5
Manipulating a database includes querying the database, updating the data,
and g


## LangChain Document + metadata Experiment

- Thực nghiệm chunking trên text có metadata

In [9]:
from langchain_core.documents import Document

# Giả lập document theo schema sau:
doc = Document(
    page_content=text,

    metadata={
        "chapter_number": "1",
        "chapter_title": "Databases and Database Users",
        "section_number": "1.1",
        "section_title": "Introduction",
        "pages": [34, 35]
    }
)

# Document bao gồm 2 phần: page_content và metadata. page_content là nội dung văn bản, metadata là thông tin về văn bản đó.

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50
)

docs = splitter.split_documents([doc])

for i, d in enumerate(docs):
    print("=" * 80)
    print("CHUNK:", i)
    print("TEXT:", d.page_content)
    print("METADATA:", d.metadata)

CHUNK: 0
TEXT: A database is a collection of related data. By data, we mean known facts
that can be recorded and that have implicit meaning.
METADATA: {'chapter_number': '1', 'chapter_title': 'Databases and Database Users', 'section_number': '1.1', 'section_title': 'Introduction', 'pages': [34, 35]}
CHUNK: 1
TEXT: A database management system (DBMS) is a computerized system that enables
users to create and maintain a database. The DBMS is a general-purpose
software system that facilitates the processes of defining, constructing,
METADATA: {'chapter_number': '1', 'chapter_title': 'Databases and Database Users', 'section_number': '1.1', 'section_title': 'Introduction', 'pages': [34, 35]}
CHUNK: 2
TEXT: manipulating, and sharing databases among various users and applications.
METADATA: {'chapter_number': '1', 'chapter_title': 'Databases and Database Users', 'section_number': '1.1', 'section_title': 'Introduction', 'pages': [34, 35]}
CHUNK: 3
TEXT: Defining a database involves specifying t

# Experiment in domain

- Fixed-size chunking + Overlap: toàn corpus -> RecursiveCharacterTextSplitte
- Structure-aware chunking: chapter/section -> RecursiveCharacterTextSplitte

## Experiment 0

In [11]:
# Mục tiêu cell: nạp dữ liệu parser, chọn Chapter 1 và dựng đơn vị section
# dùng chung cho cả ba chiến lược để phép so sánh được công bằng.
import json
import hashlib

from pathlib import Path
from collections import OrderedDict

from langchain_text_splitters import RecursiveCharacterTextSplitter

PROJECT_ROOT = Path.cwd().parent

INPUT_PATH = PROJECT_ROOT / "output" / "parsed_chapter1.jsonl"

records = []

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Total parser records:", len(records))

chunkable_records = [
    r for r in records
    if r.get("text", "").strip()
    and r.get("type") != "footnote"
]

print("Chunkable records:", len(chunkable_records))

Total parser records: 777
Chunkable records: 752


In [12]:
import pandas as pd
df = pd.DataFrame(records)

print("Kích thước:", df.shape)
print("\nCác trường hiện có:")
print(df.columns.tolist())

display(df.head(3))

Kích thước: (777, 11)

Các trường hiện có:
['id', 'pdf_page', 'chapter_number', 'chapter_title', 'section_number', 'section_title', 'type', 'text', 'raw_text', 'bbox', 'metadata']


,id,pdf_page,chapter_number,chapter_title,section_number,section_title,type,text,raw_text,bbox,metadata
0,863264e73f7a57ea:p34:b1,34,1,Databases and Database Users,NaN,Chapter Introduction,normal_text,"store images, audio clips, and video streams d...","store images, audio clips, and video streams d...","[116.99996948242188, 62.83780288696289, 467.47...",{'footnote_refs': []}
1,863264e73f7a57ea:p34:b2,34,1,Databases and Database Users,NaN,Chapter Introduction,normal_text,To understand the fundamentals of database tec...,To understand the fundamentals of database tec...,"[117.00106811523438, 176.84902954101562, 467.6...",{'footnote_refs': []}
2,863264e73f7a57ea:p34:b3,34,1,Databases and Database Users,NaN,Chapter Introduction,normal_text,The reader who desires a quick introduction to...,The reader who desires a quick introduction to...,"[117.00106811523438, 290.8580017089844, 467.44...",{'footnote_refs': []}


In [13]:
import pandas as pd

df = pd.DataFrame(records)

print("Số records:", len(df))
print("\nCác loại content:")

display(
    df["type"]
    .value_counts(dropna=False)
    .rename_axis("type")
    .reset_index(name="count")
)

Số records: 777

Các loại content:


,type,count
0,normal_text,669
1,table_like,58
2,footnote,25
3,figure_caption,25


In [14]:
required_fields = ['id', 'pdf_page', 'chapter_number', 'chapter_title', 'section_number', 'section_title', 'type', 'text', 'raw_text', 'bbox', 'metadata']
missing_fields = [
    field for field in required_fields
    if field not in df.columns
]

if missing_fields:
    print("Thiếu các trường:")
    for field in missing_fields:
        print("-", field)
else:
    print("Có đủ các trường bắt buộc")

Có đủ các trường bắt buộc


In [15]:
if "text" in df.columns:
    empty_text_mask = (
        df["text"].isna()
        | df["text"].astype(str).str.strip().eq("")
    )

    empty_text_count = empty_text_mask.sum()

    print("Số record có text rỗng:", empty_text_count)

    if empty_text_count > 0:
        display(df.loc[empty_text_mask].head(10))

Số record có text rỗng: 0


In [16]:
page_errors = pd.DataFrame()

if {"page_start", "page_end"}.issubset(df.columns):
    page_errors = df[
        df["page_start"].isna()
        | df["page_end"].isna()
        | (df["page_start"] > df["page_end"])
    ]

    print("Số record lỗi page:", len(page_errors))

    if not page_errors.empty:
        display(
            page_errors[
                ["chapter_id", "section_id",
                 "page_start", "page_end"]
            ].head(10)
        )

In [17]:
if {"section_id", "chapter_id"}.issubset(df.columns):
    section_chapter_count = (
        df.groupby("section_id")["chapter_id"]
        .nunique()
    )

    invalid_sections = section_chapter_count[
        section_chapter_count > 1
    ]

    print(
        "Số section_id xuất hiện trong nhiều chapter:",
        len(invalid_sections)
    )

    if not invalid_sections.empty:
        display(invalid_sections)

In [18]:
if "text" in df.columns:
    df["character_count"] = (
        df["text"]
        .fillna("")
        .astype(str)
        .str.len()
    )

summary = {
    "records": len(df),
    "chapters": (
        df["chapter_id"].nunique()
        if "chapter_id" in df.columns else None
    ),
    "sections": (
        df["section_id"].nunique()
        if "section_id" in df.columns else None
    ),
    "pages_min": (
        df["page_start"].min()
        if "page_start" in df.columns else None
    ),
    "pages_max": (
        df["page_end"].max()
        if "page_end" in df.columns else None
    ),
    "mean_characters": (
        round(df["character_count"].mean(), 2)
        if "character_count" in df.columns else None
    ),
    "median_characters": (
        df["character_count"].median()
        if "character_count" in df.columns else None
    )
}

summary

{'records': 777,
 'chapters': None,
 'sections': None,
 'pages_min': None,
 'pages_max': None,
 'mean_characters': np.float64(204.96),
 'median_characters': np.float64(71.0)}

In [19]:
columns_to_show = [
    col for col in [
        "chapter_id",
        "chapter_title",
        "section_id",
        "section_title",
        "page_start",
        "page_end",
        "content_type",
        "character_count",
        "text"
    ]
    if col in df.columns
]

sample_df = df[columns_to_show].copy()

if "text" in sample_df.columns:
    sample_df["text"] = sample_df["text"].str[:300]

display(sample_df.head(10))

,chapter_title,section_title,character_count,text
0,Databases and Database Users,Chapter Introduction,691,"store images, audio clips, and video streams d..."
1,Databases and Database Users,Chapter Introduction,712,To understand the fundamentals of database tec...
2,Databases and Database Users,Chapter Introduction,176,The reader who desires a quick introduction to...
3,Databases and Database Users,Introduction,445,Databases and database technology have had a m...
4,Databases and Database Users,Introduction,551,A database is a collection of related data. By...
5,Databases and Database Users,Introduction,170,The preceding definition of database is quite ...
6,Databases and Database Users,Introduction,221,We will use the word data as both singular and...
7,Databases and Database Users,Introduction,145,"constitute a database. However, the common use..."
8,Databases and Database Users,Introduction,75,■ A database represents some aspect of the rea...
9,Databases and Database Users,Introduction,101,miniworld or the universe of discourse (UoD). ...


In [20]:
from IPython.display import display

sample_columns = [
    "id",
    "pdf_page",
    "chapter_number",
    "section_number",
    "type",
    "text",
    "bbox"
]

for content_type in df["type"].dropna().unique():
    print(f"\n===== {content_type} =====")

    sample = df[df["type"] == content_type][sample_columns].head(5).copy()
    sample["text"] = sample["text"].fillna("").str[:500]

    display(sample)


===== normal_text =====


,id,pdf_page,chapter_number,section_number,type,text,bbox
0,863264e73f7a57ea:p34:b1,34,1,NaN,normal_text,"store images, audio clips, and video streams d...","[116.99996948242188, 62.83780288696289, 467.47..."
1,863264e73f7a57ea:p34:b2,34,1,NaN,normal_text,To understand the fundamentals of database tec...,"[117.00106811523438, 176.84902954101562, 467.6..."
2,863264e73f7a57ea:p34:b3,34,1,NaN,normal_text,The reader who desires a quick introduction to...,"[117.00106811523438, 290.8580017089844, 467.44..."
3,863264e73f7a57ea:p34:b5,34,1,1.1,normal_text,Databases and database technology have had a m...,"[117.0, 373.8373718261719, 467.4437255859375, ..."
4,863264e73f7a57ea:p34:b6,34,1,1.1,normal_text,A database is a collection of related data. By...,"[117.0, 449.5342712402344, 467.5617980957031, ..."



===== footnote =====


,id,pdf_page,chapter_number,section_number,type,text,bbox
6,863264e73f7a57ea:p34:fn1:b8,34,1,1.1,footnote,We will use the word data as both singular and...,"[117.0, 582.6988525390625, 461.0915832519531, ..."
25,863264e73f7a57ea:p36:fn2:b9,36,1,1.1,footnote,"The term query, originally meaning a question ...","[117.0, 592.962890625, 455.8587646484375, 612...."
39,863264e73f7a57ea:p37:fn3:b3,37,1,1.2,footnote,We use the term file informally here. At a con...,"[66.0, 592.962890625, 404.3211975097656, 612.7..."
315,863264e73f7a57ea:p61:fn1:b9,61,1,1.9,footnote,"As we shall see in Section 2.5, there are vari...","[66.0, 587.6167602539062, 399.6731872558594, 5..."
320,863264e73f7a57ea:p62:fn2:b6,62,2,2.1,footnote,Sometimes the word model is used to denote a s...,"[117.0, 558.8848876953125, 463.04437255859375,..."



===== figure_caption =====


,id,pdf_page,chapter_number,section_number,type,text,bbox
38,863264e73f7a57ea:p37:b14,37,1,1.2,figure_caption,Figure 1.1 A simplified database system enviro...,"[413.9981994628906, 334.6628112792969, 496.879..."
71,863264e73f7a57ea:p38:b32,38,1,1.2,figure_caption,Figure 1.2 A database that stores student and ...,"[29.998992919921875, 558.8836059570312, 117.39..."
97,863264e73f7a57ea:p41:b4,41,1,1.3.1,figure_caption,Figure 1.3 An example of a database catalog fo...,"[414.0, 303.72259521484375, 493.3349304199219,..."
132,863264e73f7a57ea:p43:b13,43,1,1.3.4,figure_caption,Figure 1.4 Internal storage format for a STUDE...,"[414.0, 84.89249420166016, 503.44281005859375,..."
156,863264e73f7a57ea:p44:b24,44,1,1.3.4,figure_caption,Figure 1.5 Two views derived from the database...,"[29.99989128112793, 295.2366943359375, 330.473..."



===== table_like =====


,id,pdf_page,chapter_number,section_number,type,text,bbox
41,863264e73f7a57ea:p38:b2,38,1,1.2,table_like,Smith 17 1 CS,"[152.49400329589844, 92.42674255371094, 341.34..."
42,863264e73f7a57ea:p38:b3,38,1,1.2,table_like,Brown 8 2 CS,"[152.49400329589844, 106.92698669433594, 341.3..."
45,863264e73f7a57ea:p38:b6,38,1,1.2,table_like,Intro to Computer Science CS1310 4 CS,"[143.49400329589844, 173.6809539794922, 413.82..."
46,863264e73f7a57ea:p38:b7,38,1,1.2,table_like,Data Structures CS3320 4 CS,"[143.49400329589844, 188.1809539794922, 413.82..."
47,863264e73f7a57ea:p38:b8,38,1,1.2,table_like,Discrete Mathematics MATH2410 3 MATH,"[143.49400329589844, 202.6809539794922, 424.72..."


In [21]:
metadata_fields = [
    "chapter_number",
    "chapter_title",
    "section_number",
    "section_title"
]

missing_by_type = (
    df.groupby("type")[metadata_fields]
    .agg(lambda column: column.isna().sum())
)

display(missing_by_type)

,chapter_number,chapter_title,section_number,section_title
type,,,,
figure_caption,0,0,0,0
footnote,0,0,1,0
normal_text,0,0,8,0
table_like,0,0,0,0


In [22]:
print("Tổng records:", len(df))
print("Trang nhỏ nhất:", df["pdf_page"].min())
print("Trang lớn nhất:", df["pdf_page"].max())
print("Số trang có dữ liệu:", df["pdf_page"].nunique())
print("Số chapter:", df["chapter_number"].nunique())
print("Số section:", df["section_number"].nunique())

Tổng records: 777
Trang nhỏ nhất: 34
Trang lớn nhất: 100
Số trang có dữ liệu: 67
Số chapter: 3
Số section: 58


In [23]:
df["character_count"] = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.len()
)

size_by_type = (
    df.groupby("type")["character_count"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(2)
)

display(size_by_type)

,count,min,median,mean,max
type,,,,,
figure_caption,25,11,81.0,146.32,499
footnote,25,50,160.0,160.04,282
normal_text,669,1,74.0,225.06,1572
table_like,58,5,19.0,17.76,41


# Experiment 1: Fixed-size vs Structure-aware trên toàn Chapter 1

Experiment này chỉ đánh giá tầng **ingestion/chunking**, chưa đánh giá embedding hay retrieval.

- **A — Strict fixed-size + overlap:** sliding window chạy xuyên toàn chapter, không quan tâm cấu trúc.
- **B — Global recursive:** RecursiveCharacterTextSplitter chạy trên toàn chapter; đây là control giải thích vì sao experiment cũ gần như trùng với structure-aware.
- **C — Structure-aware recursive:** chia theo section trước, sau đó mới chạy cùng recursive splitter trong từng section.

Đơn vị của chunk_size và overlap trong notebook này là **character**, không phải token.

In [24]:
import json
from collections import OrderedDict
from pathlib import Path

import pandas as pd
from IPython.display import display
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1) Tìm file đầu vào. Cell chạy được khi kernel đứng ở project root
# hoặc thư mục notebooks/, tránh phụ thuộc vào một working directory cố định.
cwd = Path.cwd()
PROJECT_ROOT = next(
    (
        root for root in (cwd, cwd.parent)
        if (root / "output" / "parsed_chapter1.jsonl").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy output/parsed_chapter1.jsonl")

INPUT_PATH = PROJECT_ROOT / "output" / "parsed_chapter1.jsonl"
# Mỗi dòng JSONL là một content record do TextbookParser tạo ra.
with INPUT_PATH.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

# 2) Giữ đúng dữ liệu sẽ được chunk: record phải có text và không phải footnote.
# Việc loại footnote ở đây giúp metric chỉ phản ánh phần nội dung chính.
chunkable_records = [
    record for record in records
    if str(record.get("text", "")).strip()
    and record.get("type") != "footnote"
]
chunk_df = pd.DataFrame(chunkable_records)

# 3) Cô lập Chapter 1 để ba chiến lược nhận cùng một tập record.
chapter_number = "1"
chapter_df = chunk_df[chunk_df["chapter_number"].astype(str).eq(chapter_number)].copy()
if chapter_df.empty:
    raise ValueError("Không tìm thấy Chapter 1")

chapter_title = str(chapter_df.iloc[0]["chapter_title"])

# 4) Chuẩn hóa section. Phần mở đầu chapter vốn có section_number = NaN
# được gán nhãn *.intro để không bị rơi khỏi phép group theo section.
chapter_df["_section_number"] = chapter_df["section_number"].apply(
    lambda value: (
        f"{chapter_number}.intro" if pd.isna(value) else str(value)
    )
)
section_numbers = chapter_df["_section_number"].drop_duplicates().tolist()

# Suy ra quan hệ phân cấp: 1.3.1 có parent là 1.3; section cấp đầu
# và phần intro có parent là chapter. Metadata này sẽ được kiểm tra ở cuối.
def parent_section_id(section_number):
    if section_number.endswith(".intro"):
        return f"chapter_{chapter_number}"
    parts = section_number.split(".")
    return (
        f"chapter_{chapter_number}"
        if len(parts) <= 2
        else f"sec_{'.'.join(parts[:-1])}"
    )

# 5) Gộp các record của cùng section theo đúng thứ tự xuất hiện.
# section_texts là đầu vào văn bản; section_metadata là ngữ cảnh đi kèm;
# section_rows chỉ phục vụ bảng thống kê để kiểm tra dữ liệu trước thí nghiệm.
section_texts = OrderedDict()
section_metadata = OrderedDict()
section_rows = []

for section_number in section_numbers:
    current = chapter_df[
        chapter_df["_section_number"].eq(section_number)
    ]
    raw_title = current.iloc[0]["section_title"]
    section_title = (
        "Chapter Introduction" if pd.isna(raw_title) else str(raw_title)
    )
    # Nối các block trong section và chèn marker dễ nhìn khi inspect chunk.
    body = "\n".join(current["text"].dropna().astype(str))
    text = f"[[SECTION {section_number}: {section_title}]]\n{body}"

    section_texts[section_number] = text
    section_metadata[section_number] = {
        "strategy": "structure_aware_recursive",
        "chapter_number": chapter_number,
        "chapter_title": chapter_title,
        "section_id": f"sec_{section_number}",
        "section_number": section_number,
        "section_title": section_title,
        "parent_section_id": parent_section_id(section_number),
    }
    section_rows.append({
        "section_number": section_number,
        "section_title": section_title,
        "records": len(current),
        "characters": len(text),
    })

# 6) Sanity check: xem số record, loại content và kích thước từng section
# trước khi chạy chunking; cell này chưa so sánh chất lượng chunk.
section_stats_df = pd.DataFrame(section_rows)

print("Chapter:", chapter_number, "-", chapter_title)
print("Records toàn file:", len(records))
print("Records dùng cho Chapter 1:", len(chapter_df))
print("Footnotes còn lại:", int((chapter_df["type"] == "footnote").sum()))
print("Structural units (intro + sections):", len(section_texts))
display(chapter_df["type"].value_counts().to_frame("count"))
display(section_stats_df)

Chapter: 1 - Databases and Database Users
Records toàn file: 777
Records dùng cho Chapter 1: 312
Footnotes còn lại: 0
Structural units (intro + sections): 34


,count
type,
normal_text,255
table_like,50
figure_caption,7


,section_number,section_title,records,characters
0,1.intro,Chapter Introduction,3,1623
1,1.1,Introduction,20,6805
2,1.2,An Example,60,5453
3,1.3,Characteristics of the Database Approach,7,1660
4,1.3.1,Self-Describing Nature of a Database System,24,2569
5,1.3.2,"Insulation between Programs and Data, and Data...",8,4400
6,1.3.3,Support of Multiple Views of the Data,1,903
7,1.3.4,Sharing of Data and Multiuser Transaction Proc...,31,2655
8,1.4,Actors on the Scene,1,732
9,1.4.1,Database Administrators,1,746


In [25]:
# Mục tiêu cell: tạo ba tập chunk A/B/C với cùng chunk_size và overlap.
# Khác biệt duy nhất cần quan sát là cách mỗi chiến lược xử lý boundary.
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
SECTION_JOINER = "\n\n"

# Nối các section thành một corpus và đồng thời lưu khoảng [start, end)
# của từng section. Các span sẽ giúp phát hiện chunk cắt qua boundary.
def build_corpus(texts, joiner):
    parts, spans, cursor = [], [], 0
    items = list(texts.items())

    for index, (section_number, text) in enumerate(items):
        start, end = cursor, cursor + len(text)
        parts.append(text)
        spans.append({
            "section_number": section_number,
            "start": start,
            "end": end,
        })
        cursor = end + (len(joiner) if index < len(items) - 1 else 0)

    corpus = joiner.join(parts)
    assert len(corpus) == cursor
    return corpus, spans

# Baseline A: sliding window ký tự thuần túy. Nó bảo đảm kích thước/overlap
# nhưng cố ý không nhìn paragraph, câu hay section để làm mốc đối chứng.
def strict_fixed_chunking(text, size, overlap, metadata=None):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Cấu hình size/overlap không hợp lệ")

    chunks, start, step = [], 0, size - overlap
    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(Document(
            page_content=text[start:end],
            metadata={
                **dict(metadata or {}),
                "strategy": "strict_fixed",
                "start_index": start,
                "end_index": end,
            },
        ))
        if end == len(text):
            break
        start += step
    return chunks

# Splitter dùng chung cho B và C. Thứ tự separators thể hiện mức ưu tiên:
# paragraph -> dòng -> câu -> từ -> ký tự (fallback cuối cùng).
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True,
)

# Corpus chuẩn nối section bằng hai newline; đây cũng là separator mạnh nhất.
corpus_text, section_spans = build_corpus(
    section_texts,
    SECTION_JOINER,
)

# A) Fixed-size thật trên toàn chapter: có thể cắt ngang section.
strict_fixed_chunks = strict_fixed_chunking(
    corpus_text,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    {
        "chapter_number": chapter_number,
        "chapter_title": chapter_title,
    },
)

# B) Control recursive trên toàn chapter: không ép boundary bằng code,
# nhưng splitter có thể tự dừng tại hai newline giữa các section.
global_recursive_chunks = recursive_splitter.create_documents(
    [corpus_text],
    [{
        "strategy": "global_recursive",
        "chapter_number": chapter_number,
        "chapter_title": chapter_title,
    }],
)

# C) Structure-aware: đưa mỗi section vào splitter như một document riêng.
# Vì vậy một chunk không thể chứa text của hai section khác nhau.
structure_aware_chunks = recursive_splitter.create_documents(
    list(section_texts.values()),
    list(section_metadata.values()),
)

# LangChain trả start_index; bổ sung end_index để đối chiếu chunk với span.
# Với structure-aware, index này là vị trí cục bộ bên trong từng section.
for document in global_recursive_chunks + structure_aware_chunks:
    document.metadata["end_index"] = (
        document.metadata["start_index"] + len(document.page_content)
    )

# Đếm nhanh số chunk để thấy việc tôn trọng cấu trúc có làm phân mảnh hơn không.
print("Chapter characters:", len(corpus_text))
print("Strict fixed chunks:", len(strict_fixed_chunks))
print("Global recursive chunks:", len(global_recursive_chunks))
print("Structure-aware chunks:", len(structure_aware_chunks))

Chapter characters: 67485
Strict fixed chunks: 80
Global recursive chunks: 102
Structure-aware chunks: 102


In [26]:
# Mục tiêu cell: đo kích thước, overlap thực tế và mức độ cắt qua section
# của ba tập chunk, thay vì chỉ nhìn vài ví dụ bằng mắt.
# Ánh xạ [start_index, end_index) của chunk sang các span section giao nhau.
def section_ids_for_chunk(document, spans):
    if document.metadata["strategy"] == "structure_aware_recursive":
        return [document.metadata["section_number"]]

    start = document.metadata["start_index"]
    end = document.metadata["end_index"]
    return [
        span["section_number"] for span in spans
        if start < span["end"] and end > span["start"]
    ]

# Tìm số ký tự cuối chunk trái trùng với đầu chunk phải. Đây là overlap
# thực tế; recursive splitter không cam kết lúc nào cũng đạt đúng mức 150.
def suffix_prefix_overlap(left, right, limit=CHUNK_OVERLAP):
    for length in range(min(len(left), len(right), limit), 0, -1):
        if left[-length:] == right[:length]:
            return length
    return 0

# Đo overlap cho từng cặp chunk liên tiếp trong một chiến lược.
def measured_overlaps(chunks):
    values = []
    for left, right in zip(chunks, chunks[1:]):
        values.append(suffix_prefix_overlap(
            left.page_content,
            right.page_content,
        ))
    return values

# Gom các tín hiệu chính: phân phối size, mức tận dụng chunk_size,
# chunk quá ngắn, chunk vượt boundary và overlap hiệu dụng.
def summarize(name, chunks, spans):
    sizes = pd.Series([len(doc.page_content) for doc in chunks])
    section_counts = [
        len(section_ids_for_chunk(doc, spans)) for doc in chunks
    ]
    overlaps = measured_overlaps(chunks)

    return {
        "strategy": name,
        "chunks": len(chunks),
        "min": int(sizes.min()),
        "p10": round(float(sizes.quantile(0.10)), 1),
        "median": round(float(sizes.median()), 1),
        "p90": round(float(sizes.quantile(0.90)), 1),
        "max": int(sizes.max()),
        "mean": round(float(sizes.mean()), 1),
        "utilization_%": round(100 * sizes.mean() / CHUNK_SIZE, 1),
        "chunks_below_300": int((sizes < 300).sum()),
        "cross_boundary_chunks": sum(x > 1 for x in section_counts),
        "cross_boundary_%": round(
            100 * sum(x > 1 for x in section_counts) / len(chunks),
            1,
        ),
        "mean_effective_overlap": (
            round(sum(overlaps) / len(overlaps), 1) if overlaps else 0.0
        ),
        "zero_overlap_pairs": sum(x == 0 for x in overlaps),
    }

# Đặt ba chiến lược cạnh nhau trong cùng một bảng để so sánh trực tiếp.
primary_results = pd.DataFrame([
    summarize("A. strict_fixed", strict_fixed_chunks, section_spans),
    summarize("B. global_recursive", global_recursive_chunks, section_spans),
    summarize(
        "C. structure_aware_recursive",
        structure_aware_chunks,
        section_spans,
    ),
])
display(primary_results)

# Kiểm tra một hiện tượng quan trọng: B và C có thể cho content giống hệt
# vì hai newline của corpus đã vô tình hoạt động như section boundary.
same_recursive_content = (
    [doc.page_content for doc in global_recursive_chunks]
    ==
    [doc.page_content for doc in structure_aware_chunks]
)
print(
    "Global recursive và structure-aware giống hệt content:",
    same_recursive_content,
)
if same_recursive_content:
    print(
        "Hai newline giữa các section là separator ưu tiên cao nhất, "
        "nên global recursive đã vô tình tôn trọng section boundary."
    )

,strategy,chunks,min,p10,median,p90,max,mean,utilization_%,chunks_below_300,cross_boundary_chunks,cross_boundary_%,mean_effective_overlap,zero_overlap_pairs
0,A. strict_fixed,80,335,1000.0,1000.0,1000.0,1000,991.7,99.2,0,30,37.5,150.0,0
1,B. global_recursive,102,50,239.7,738.0,971.9,995,685.0,68.5,13,0,0.0,24.6,77
2,C. structure_aware_recursive,102,50,239.7,738.0,971.9,995,685.0,68.5,13,0,0.0,24.6,77


Global recursive và structure-aware giống hệt content: True
Hai newline giữa các section là separator ưu tiên cao nhất, nên global recursive đã vô tình tôn trọng section boundary.


## Boundary stress test: giữ nguyên splitter, thay cách nối

Đây là controlled test để cô lập ảnh hưởng của section boundary. Các section được nối bằng một newline thay vì hai newline, khiến global recursive không còn nhận được ranh giới mạnh. Đây không phải cấu hình production được đề xuất.

In [27]:
# Mục tiêu stress test: làm yếu boundary từ hai newline xuống một newline.
# Nếu global recursive bắt đầu trộn section, ta biết kết quả B=C ở trên là
# do separator trong dữ liệu, không phải do B thật sự hiểu cấu trúc section.
flat_corpus_text, flat_section_spans = build_corpus(
    section_texts,
    "\n",
)
flat_global_recursive_chunks = recursive_splitter.create_documents(
    [flat_corpus_text],
    [{
        "strategy": "flat_global_recursive",
        "chapter_number": chapter_number,
        "chapter_title": chapter_title,
    }],
)
# Bổ sung end_index để dùng lại chính xác bộ metric của cell trước.
for document in flat_global_recursive_chunks:
    document.metadata["end_index"] = (
        document.metadata["start_index"] + len(document.page_content)
    )

# So B2 với C: chỉ C có boundary được ép bằng cách tách document đầu vào.
display(pd.DataFrame([
    summarize(
        "B2. flat_global_recursive",
        flat_global_recursive_chunks,
        flat_section_spans,
    ),
    summarize(
        "C. structure_aware_recursive",
        structure_aware_chunks,
        section_spans,
    ),
]))

,strategy,chunks,min,p10,median,p90,max,mean,utilization_%,chunks_below_300,cross_boundary_chunks,cross_boundary_%,mean_effective_overlap,zero_overlap_pairs
0,B2. flat_global_recursive,91,67,475.0,833.0,978.0,995,776.2,77.6,4,26,28.6,35.9,52
1,C. structure_aware_recursive,102,50,239.7,738.0,971.9,995,685.0,68.5,13,0,0.0,24.6,77


In [28]:
# Mục tiêu cell: inspect các lỗi boundary cụ thể và xem phân phối size
# chi tiết hơn cho cả bốn biến thể A, B, B2 và C.
# Lấy tối đa `limit` chunk chứa nhiều section để đọc preview bằng mắt.
def boundary_examples(name, chunks, spans, limit=5):
    rows = []
    for index, document in enumerate(chunks):
        section_ids = section_ids_for_chunk(document, spans)
        if len(section_ids) > 1:
            rows.append({
                "strategy": name,
                "chunk_index": index,
                "characters": len(document.page_content),
                "section_ids": section_ids,
                "preview": document.page_content.replace("\n", " ↵ ")[:500],
            })
        if len(rows) == limit:
            break
    return rows

# Hiển thị ví dụ của fixed-size và stress-test recursive, hai trường hợp
# được kỳ vọng sẽ có chunk cắt ngang section.
display(pd.DataFrame(
    boundary_examples(
        "strict_fixed",
        strict_fixed_chunks,
        section_spans,
    )
    +
    boundary_examples(
        "flat_global_recursive",
        flat_global_recursive_chunks,
        flat_section_spans,
    )
))

# Chuyển mọi chunk thành từng dòng metric để có thể group/bucket bằng pandas.
strategy_sets = [
    ("strict_fixed", strict_fixed_chunks, section_spans),
    ("global_recursive", global_recursive_chunks, section_spans),
    (
        "flat_global_recursive",
        flat_global_recursive_chunks,
        flat_section_spans,
    ),
    (
        "structure_aware_recursive",
        structure_aware_chunks,
        section_spans,
    ),
]
rows = []
for name, chunks, spans in strategy_sets:
    for index, document in enumerate(chunks):
        section_ids = section_ids_for_chunk(document, spans)
        rows.append({
            "strategy": name,
            "chunk_index": index,
            "characters": len(document.page_content),
            "section_count": len(section_ids),
            "section_ids": section_ids,
        })

# Chia size thành bốn bucket: bảng kết quả cho biết chiến lược nào tạo nhiều
# chunk nhỏ và chiến lược nào thường tận dụng gần hết giới hạn 1000 ký tự.
chunk_metrics_df = pd.DataFrame(rows)
chunk_metrics_df["size_bucket"] = pd.cut(
    chunk_metrics_df["characters"],
    [0, 300, 600, 900, CHUNK_SIZE],
    labels=["1-300", "301-600", "601-900", "901-1000"],
    include_lowest=True,
)
display(
    chunk_metrics_df
    .groupby(["strategy", "size_bucket"], observed=False)
    .size()
    .unstack(fill_value=0)
)

,strategy,chunk_index,characters,section_ids,preview
0,strict_fixed,1,1000,"[1.intro, 1.1]",e applications. In Section 1.1 we start by def...
1,strict_fixed,9,1000,"[1.1, 1.2]",", so the DBMS must be able to maintain the dat..."
2,strict_fixed,16,1000,"[1.2, 1.3]",MSs in Chapters 5 through 9.) ↵ The final stag...
3,strict_fixed,18,1000,"[1.3, 1.3.1]",pport of multiple views of the data ↵ ■ Sharin...
4,strict_fixed,21,1000,"[1.3.1, 1.3.2]",ame Character (30) STUDENT ↵ Student_number Ch...
5,flat_global_recursive,1,919,"[1.intro, 1.1]",To understand the fundamentals of database tec...
6,flat_global_recursive,10,682,"[1.1, 1.2]",It is not absolutely necessary to use general-...
7,flat_global_recursive,19,313,"[1.2, 1.3]","The final stage is physical design, during whi..."
8,flat_global_recursive,21,711,"[1.3, 1.3.1]","In the database approach, a single repository ..."
9,flat_global_recursive,25,703,"[1.3.1, 1.3.2]",Class Integer (1) STUDENT ↵ Major Major_type S...


size_bucket,1-300,301-600,601-900,901-1000
strategy,,,,
flat_global_recursive,4,11,39,37
global_recursive,13,17,40,32
strict_fixed,0,1,0,79
structure_aware_recursive,13,17,40,32


In [29]:
# Mục tiêu cell: kiểm tra metadata của C đủ để truy vết chapter/section
# và chạy các assertion bảo vệ những giả định quan trọng của experiment.
# Chỉ lấy các trường cần cho retrieval/filtering và truy vết hierarchy.
metadata_fields = [
    "chapter_number",
    "chapter_title",
    "section_id",
    "section_number",
    "section_title",
    "parent_section_id",
    "start_index",
    "end_index",
]
metadata_sample_df = pd.DataFrame([
    {key: doc.metadata.get(key) for key in metadata_fields}
    for doc in structure_aware_chunks
])
display(metadata_sample_df.head(10))

# Dùng section lồng 1.3.1 làm ví dụ để xác nhận parent_section_id = sec_1.3.
sample_131 = metadata_sample_df[metadata_sample_df["section_number"].astype(str).eq("1.3.1")].head(1)

if not sample_131.empty:
    print("Hierarchy mẫu: parent của section 1.3.1 phải là sec_1.3")
    display(sample_131)

# Sanity checks: không footnote/rỗng, C không vượt section, A có vượt section,
# và mọi chunk structure-aware đều giữ đủ metadata cấu trúc.
assert int((chapter_df["type"] == "footnote").sum()) == 0
assert all(doc.page_content for doc in strict_fixed_chunks)
assert all(doc.page_content for doc in global_recursive_chunks)
assert all(doc.page_content for doc in structure_aware_chunks)
assert all(len(section_ids_for_chunk(doc, section_spans)) == 1 for doc in structure_aware_chunks)
assert any(len(section_ids_for_chunk(doc, section_spans)) > 1 for doc in strict_fixed_chunks)
assert metadata_sample_df[["section_id", "section_number", "section_title", "parent_section_id"]].notna().all().all()

print("Sanity checks passed.")

,chapter_number,chapter_title,section_id,section_number,section_title,parent_section_id,start_index,end_index
0,1,Databases and Database Users,sec_1.intro,1.intro,Chapter Introduction,chapter_1,0,733
1,1,Databases and Database Users,sec_1.intro,1.intro,Chapter Introduction,chapter_1,734,1623
2,1,Databases and Database Users,sec_1.1,1.1,Introduction,chapter_1,0,475
3,1,Databases and Database Users,sec_1.1,1.1,Introduction,chapter_1,476,1420
4,1,Databases and Database Users,sec_1.1,1.1,Introduction,chapter_1,1345,1868
5,1,Databases and Database Users,sec_1.1,1.1,Introduction,chapter_1,1763,2479
6,1,Databases and Database Users,sec_1.1,1.1,Introduction,chapter_1,2480,3466
7,1,Databases and Database Users,sec_1.1,1.1,Introduction,chapter_1,3467,4462
8,1,Databases and Database Users,sec_1.1,1.1,Introduction,chapter_1,4463,5438
9,1,Databases and Database Users,sec_1.1,1.1,Introduction,chapter_1,5439,6150


Hierarchy mẫu: parent của section 1.3.1 phải là sec_1.3


,chapter_number,chapter_title,section_id,section_number,section_title,parent_section_id,start_index,end_index
23,1,Databases and Database Users,sec_1.3.1,1.3.1,Self-Describing Nature of a Database System,sec_1.3,0,955


Sanity checks passed.


## Cách đọc kết quả ingestion

- **cross_boundary_chunks:** số chunk trộn nhiều section. Strict fixed-size thường lớn hơn 0; structure-aware phải bằng 0.
- **utilization_%:** mức sử dụng kích thước mục tiêu. Fixed-size thường cao hơn; structure-aware có thể tạo thêm chunk nhỏ ở cuối section.
- **chunks_below_300:** cảnh báo fragmentation do section ngắn hoặc phần dư cuối section.
- **mean_effective_overlap:** overlap thực tế. Với recursive splitter, chunk_overlap=150 là giới hạn mong muốn chứ không bảo đảm mọi cặp overlap đúng 150.
- Global recursive giống structure-aware trong dữ liệu hiện tại là kết quả hợp lệ: hai newline đang đóng vai trò như section boundary ngầm.
- Flat global recursive chỉ là stress test chứng minh tác động của boundary, không phải đề xuất xoá cấu trúc đầu vào.

Các metric này phù hợp cho bước ingestion. Sau khi chọn policy chunking, có thể đóng gói thành class rồi chuyển sang notebook embedding/retrieval riêng.